In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))
from src.data.inventory import build_reports_table

reports_df = build_reports_table()


In [22]:
reports_df

,report_id,comparison,indication,findings,impression,image_ids,n_images,mesh_major,mesh_automatic
0,CXR1,None.,Positive TB test,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,"[CXR1_1_IM-0001-3001, CXR1_1_IM-0001-4001]",2,[normal],[]
1,CXR10,Chest radiographs XXXX.,"XXXX-year-old male, chest pain.",The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.,"[CXR10_IM-0002-1001, CXR10_IM-0002-2001]",2,[Calcified Granuloma/lung/upper lobe/right],[calcified granuloma]
2,CXR100,None.,NaN,Both lungs are clear and expanded. Heart and m...,No active disease.,"[CXR100_IM-0002-1001, CXR100_IM-0002-2001]",2,[normal],[]
3,CXR1000,XXXX PA and lateral chest radiographs,"XXXX-year-old male, XXXX.",There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...,"[CXR1000_IM-0003-1001, CXR1000_IM-0003-2001, C...",3,"[Opacity/lung/upper lobe/right, Pulmonary Atel...","[atelectases, mass lesion, opacity, Atelectasi..."
4,CXR1001,None,"dyspnea, subjective fevers, arthritis, immigra...",Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.,"[CXR1001_IM-0004-1001, CXR1001_IM-0004-1002]",2,[Markings/lung/bilateral/interstitial/diffuse/...,"[diffuse fibrosis, Fibrosis, Pulmonary Fibrosis]"
...,...,...,...,...,...,...,...,...,...
3950,CXR995,"XXXX, XXXX.",SP CABG SOB NO RALES,Sternotomy sutures and bypass grafts have been...,Post operative chest with no acute disease.,"[CXR995_IM-2478-1001, CXR995_IM-2478-1002]",2,[Sutures/sternum],"[bypass grafts, sternotomy, Coronary Artery By..."
3951,CXR996,NaN,chest pain.,NaN,Heart size is normal and lungs are clear. No p...,"[CXR996_IM-2479-1001, CXR996_IM-2479-2001]",2,[normal],[]
3952,CXR997,Chest x-XXXX dated XXXX at XXXX hours.,"XXXX-year-old female, transplant workup.",Calcified mediastinal XXXX. No focal areas of ...,No acute cardiopulmonary abnormality. .,"[CXR997_IM-2479-1001, CXR997_IM-2479-2001]",2,"[Calcinosis/mediastinum, Thoracic Vertebrae/de...",[degenerative change]
3953,CXR998,None.,XXXX-year-old XXXX with pain.,Cardiomediastinal silhouette demonstrates norm...,No acute cardiopulmonary abnormality.,"[CXR998_IM-2479-1001, CXR998_IM-2479-2001]",2,"[Atherosclerosis/aorta, thoracic, Aorta, Thora...","[atheroscleroses, degenerative disc diseases, ..."


In [23]:
mesh_majors = reports_df['mesh_major'].explode()
#explode gives each list item its own row and repeats the row label
mesh_majors.value_counts()

mesh_major
normal                                      1391
Lung/hypoinflation                           245
Lung/hyperdistention                         164
Cardiomegaly                                 157
Cardiomegaly/mild                            135
                                            ... 
Opacity/lung/middle lobe/right/reticular       1
Bronchiectasis/middle lobe/right               1
Opacity/lung/base/bilateral/scattered          1
Opacity/lung/hilum/streaky/mild                1
Sutures/sternum                                1
Name: count, Length: 1719, dtype: int64

In [24]:
list(reports_df['mesh_major'].explode())[0:14] #these are not the top 14
#e ranking is already in mesh_majors, because value_counts() 
# sorts from most to least frequent. The notebook shows only its first and
#  last 5 rows because it has more than 60 rows.

['normal',
 'Calcified Granuloma/lung/upper lobe/right',
 'normal',
 'Opacity/lung/upper lobe/right',
 'Pulmonary Atelectasis/upper lobe/right',
 'Opacity/lung/lingula',
 'Markings/lung/bilateral/interstitial/diffuse/prominent',
 'Fibrosis/diffuse',
 'Mastectomy/left',
 'Density/retrocardiac',
 'Calcinosis/blood vessels',
 'Calcified Granuloma',
 'Opacity/lung/base/left',
 'Bone Diseases, Metabolic/spine']

In [25]:
mesh_majors.str.split('/').str[0].value_counts()

mesh_major
normal                   1391
Lung                      568
Opacity                   536
Cardiomegaly              375
Pulmonary Atelectasis     347
                         ... 
Bronchiolitis               1
Blood Vessels               1
Hemothorax                  1
Trachea                     1
Mediastinal Emphysema       1
Name: count, Length: 122, dtype: int64

Why we need labels at all. In the course, every training example was a pair (x, y): x was the image and y the answer, for example cat = 1 or not cat = 0. In Phase 1 we train a classifier the same way:

x = an X-ray image.
y = 14 yes/no answers, one for each finding: "Is there cardiomegaly? Is there pleural effusion? …"
The dataset doesn't give us y. What it gives is the MeSH terms: 1,719 different, messy strings like Cardiomegaly/mild. A network needs a fixed number of outputs, and each position must always mean the same thing. So we have to translate each report's terms into those 14 answers.

In [26]:
from src.data.labels import split_term

print(split_term("Costophrenic Angle/right/blunted/mild"))   # ['costophrenic angle', 'right', 'blunted', 'mild']
print(split_term("Mediastinum/paratracheal/enlarged "))      # ['mediastinum', 'paratracheal', 'enlarged']
print(split_term("Subcutaneous  Emphysema"))                 # ['subcutaneous emphysema']
print(split_term("normal"))                                  # ['normal']


['costophrenic angle', 'right', 'blunted', 'mild']
['mediastinum', 'paratracheal', 'enlarged']
['subcutaneous emphysema']
['normal']


In [27]:
from src.data.labels import term_matches

print(term_matches("Cardiomegaly/mild", "Cardiomegaly"))                                # True   plain head
print(term_matches("Cardiac Shadow/borderline", "Cardiac Shadow/enlarged"))             # False  word missing
print(term_matches("Costophrenic Angle/right/blunted/mild", "Costophrenic Angle/blunted"))  # True   word later in the term
print(term_matches("Cystic Fibrosis", "Fibrosis"))                                      # False  heads differ
print(term_matches("Thoracic vertebrae/degenerative", "Thoracic Vertebrae"))            # True   case ignored
print(term_matches("Cysts/humerus/left", "Cysts/lung"))                                 # False  the bone cyst


True
False
True
False
True
False


In [28]:
from src.data.labels import report_labels
from src.data.splits import load_config
from src.data.inventory import PROJECT_ROOT

label_map = load_config(PROJECT_ROOT / "configs" / "label_map.yaml")

tests = {
    "CXR1": ["normal"],
    "CXR1230": ["Cardiomegaly/mild", "Pleural Effusion/right", "Infiltrate/lung/lower lobe/right", "Hydropneumothorax/right"],
    "two Opacity terms": ["Opacity/lung/upper lobe/right", "Opacity/lung/lingula"],
    "CXR26": ["Spondylosis/thoracic vertebrae"],
}
for name, terms in tests.items():
    labels = report_labels(terms, label_map)
    print(name, len(labels), [c for c, v in labels.items() if v == 1])


CXR1 14 ['No Finding']
CXR1230 14 ['Cardiomegaly', 'Lung Opacity', 'Pneumothorax', 'Pleural Effusion']
two Opacity terms 14 ['Lung Opacity']
CXR26 14 []


In [29]:
reports_df

,report_id,comparison,indication,findings,impression,image_ids,n_images,mesh_major,mesh_automatic
0,CXR1,None.,Positive TB test,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,"[CXR1_1_IM-0001-3001, CXR1_1_IM-0001-4001]",2,[normal],[]
1,CXR10,Chest radiographs XXXX.,"XXXX-year-old male, chest pain.",The cardiomediastinal silhouette is within nor...,No acute cardiopulmonary process.,"[CXR10_IM-0002-1001, CXR10_IM-0002-2001]",2,[Calcified Granuloma/lung/upper lobe/right],[calcified granuloma]
2,CXR100,None.,NaN,Both lungs are clear and expanded. Heart and m...,No active disease.,"[CXR100_IM-0002-1001, CXR100_IM-0002-2001]",2,[normal],[]
3,CXR1000,XXXX PA and lateral chest radiographs,"XXXX-year-old male, XXXX.",There is XXXX increased opacity within the rig...,1. Increased opacity in the right upper lobe w...,"[CXR1000_IM-0003-1001, CXR1000_IM-0003-2001, C...",3,"[Opacity/lung/upper lobe/right, Pulmonary Atel...","[atelectases, mass lesion, opacity, Atelectasi..."
4,CXR1001,None,"dyspnea, subjective fevers, arthritis, immigra...",Interstitial markings are diffusely prominent ...,Diffuse fibrosis. No visible focal acute disease.,"[CXR1001_IM-0004-1001, CXR1001_IM-0004-1002]",2,[Markings/lung/bilateral/interstitial/diffuse/...,"[diffuse fibrosis, Fibrosis, Pulmonary Fibrosis]"
...,...,...,...,...,...,...,...,...,...
3950,CXR995,"XXXX, XXXX.",SP CABG SOB NO RALES,Sternotomy sutures and bypass grafts have been...,Post operative chest with no acute disease.,"[CXR995_IM-2478-1001, CXR995_IM-2478-1002]",2,[Sutures/sternum],"[bypass grafts, sternotomy, Coronary Artery By..."
3951,CXR996,NaN,chest pain.,NaN,Heart size is normal and lungs are clear. No p...,"[CXR996_IM-2479-1001, CXR996_IM-2479-2001]",2,[normal],[]
3952,CXR997,Chest x-XXXX dated XXXX at XXXX hours.,"XXXX-year-old female, transplant workup.",Calcified mediastinal XXXX. No focal areas of ...,No acute cardiopulmonary abnormality. .,"[CXR997_IM-2479-1001, CXR997_IM-2479-2001]",2,"[Calcinosis/mediastinum, Thoracic Vertebrae/de...",[degenerative change]
3953,CXR998,None.,XXXX-year-old XXXX with pain.,Cardiomediastinal silhouette demonstrates norm...,No acute cardiopulmonary abnormality.,"[CXR998_IM-2479-1001, CXR998_IM-2479-2001]",2,"[Atherosclerosis/aorta, thoracic, Aorta, Thora...","[atheroscleroses, degenerative disc diseases, ..."


In [36]:
from src.data.labels import build_labels_table

labels_df = build_labels_table(reports_df, label_map)
print(labels_df.shape)
print(labels_df["labeled"].value_counts())
labels_df.head(3)


(3955, 16)
labeled
1    3860
0      95
Name: count, dtype: int64


,report_id,labeled,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,Lung Lesion,Edema,Consolidation,Pneumonia,Atelectasis,Pneumothorax,Pleural Effusion,Pleural Other,Fracture,Support Devices
0,CXR1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
1,CXR10,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,CXR100,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
from src.data.splits import load_config
LABEL_MAP_PATH = PROJECT_ROOT / "configs" / "label_map.yaml"
cfg = load_config(path=LABEL_MAP_PATH)
cfg


In [40]:
list(cfg)

['No Finding',
 'Enlarged Cardiomediastinum',
 'Cardiomegaly',
 'Lung Opacity',
 'Lung Lesion',
 'Edema',
 'Consolidation',
 'Pneumonia',
 'Atelectasis',
 'Pneumothorax',
 'Pleural Effusion',
 'Pleural Other',
 'Fracture',
 'Support Devices']

## Per-split check: are the categories spread evenly over train / val / test?

The split (Step 0.2) was random by report, without looking at labels. Here we count each category in each split,
using only labeled reports. Result and discussion: `reports/labels.md`, section 6.

In [ ]:
from src.data.splits import INTERIM_DIR

categories = list(label_map)
counts, percents = {}, {}

for name in ["train", "val", "test"]:
    split_ids = pd.read_csv(INTERIM_DIR / f"{name}.csv")["report_id"].unique()
    split_labels = labels_df[labels_df["report_id"].isin(split_ids)]
    labeled = split_labels[split_labels["labeled"] == 1]
    counts[name] = labeled[categories].sum()
    percents[name] = (counts[name] / len(labeled) * 100).round(1)
    print(f"{name}: {len(labeled)} labeled reports")

per_split = pd.concat({"count": pd.DataFrame(counts), "percent": pd.DataFrame(percents)}, axis=1)
per_split.sort_values(("count", "train"), ascending=False)